In [29]:
using ProgressMeter

include("../main.jl");

Test Summary: | Pass  Total  Time
dyck          |    3      3  0.1s
Test Summary: | Pass  Total  Time
forest        |    4      4  0.0s
Test Summary: | Pass  Total  Time
merge         |    2      2  0.1s
Test Summary: | Pass  Total  Time
CellComplex   |    3      3  0.1s
Test Summary: | Pass  Total  Time
isNWT         |    3      3  0.1s
Test Summary: | Pass  Total  Time
remove_edges  |    4      4  0.6s
Test Summary: | Pass  Total  Time
nonline       |    3      3  0.0s


In [30]:
viro_dict = Dict{NWT,Tuple{Int,Vector{Bool}}}()
filename = "cubic_triangulations.csv"
progress = Progress(countlines(filename); dt=1.0)
open(filename, "r") do io
    for (itriang, line) in enumerate(eachline(io))
        m = match(r"\{(.*)\}\{(.*)\}\{(.*)\}", line)

        vertices = Tuple{Int,Int}[]
        for pt in eachmatch(r"\{(.+?)\}", m.captures[1])
            push!(vertices, tuple(parse.(Int, split(pt.captures[1], ","))...))
        end

        triang = Tuple{Int,Int,Int}[]
        for tr in eachmatch(r"\{(.+?)\}", m.captures[2])
            push!(triang, tuple(parse.(Int, split(tr.captures[1], ","))...))
        end

        viro_patchworking!(3, itriang, vertices, triang, viro_dict)
        next!(progress)
    end
end

for _ in 1:2
    for (nwt, (itriang, sign)) in copy(viro_dict)
        translated = translate_up(nwt)
        if !haskey(viro_dict, translated)
            for symm in symmetries(translated)
                viro_dict[symm] = (itriang, sign)
            end
        end
    end
end

open(GzipCompressorStream, "cubic_viro.jsonl.gz", "w") do out_io
    write_cc_to_jsonl!(out_io, lines_xyz)
    for (nwt, (itriang, sign)) in viro_dict
        write_nwt_to_jsonl!(out_io, nwt, itriang, sign)
    end
end

viro = collect(keys(viro_dict))

Progress: 100%|█████████████████████████████████████████| Time: 0:00:02


2024-element Vector{NWT}:
 ([3, 3, 2, 0, 0, 1], Vector{Bool}[[1, 1, 0, 1, 0, 1, 0, 0], [1, 1, 0, 0], [1, 1, 0, 0], [1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 0, 1, 0, 3, 0], Vector{Bool}[[1, 0, 1, 0], [1, 0, 1, 0, 1, 0], [], [1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([1, 2, 1, 2, 1, 2], Vector{Bool}[[1, 0, 1, 0], [1, 0, 1, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 3, 0, 0, 0, 3], Vector{Bool}[[1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], []], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([2, 3, 1, 1, 0, 2], Vector{Bool}[[1, 1, 0, 0, 1, 0], [1, 1, 0, 0], [1, 0, 1, 0, 1, 0], [1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 8, T = [1, 0])])
 ([1, 1, 0, 2, 2, 3], Vector{Bool}[[1, 0], [1, 1, 0, 0, 1, 0], [1, 1, 0, 1, 0, 0], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 8, T = [1, 0])])
 ([3, 3, 0, 0, 0, 3], Vector{Bool}[[1, 1, 1, 0, 0, 0], [1, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0], []], @Nam

In [31]:
floatless = all_floatless_bezout_zero(lines_xyz, [1, 1, 1, 3])

1240-element Vector{NWT}:
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 1, 0, 1, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 1, 0, 0, 1, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 0, 1, 1, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 0, 1, 0, 1, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([0, 0, 0, 3, 3, 3], Vector{Bool}[[], [1, 1, 0, 1, 0, 0], [1, 1, 0, 1, 0, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, 

In [32]:
any([is_bezout_order_one(nwt, [1,1,1,3]) for nwt in remove_symmetries(setdiff(floatless, viro), symmetries)])

false

In [33]:
bezout = NWT[]
for nwt in [nwt for nwt in viro if isempty(nwt.T)]
    append!(bezout, all_nonnested_floating(nwt, [1,1,1,3]))
end

open(GzipCompressorStream, "cubic_bezout.jsonl.gz", "w") do out_io
    write_cc_to_jsonl!(out_io, lines_xyz)
    for nwt in bezout
        write_nwt_to_jsonl!(out_io, nwt, 0, [])
    end
end

bezout

2072-element Vector{NWT}:
 ([3, 3, 2, 0, 0, 1], Vector{Bool}[[1, 1, 0, 1, 0, 1, 0, 0], [1, 1, 0, 0], [1, 1, 0, 0], [1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 0, 1, 0, 3, 0], Vector{Bool}[[1, 0, 1, 0], [1, 0, 1, 0, 1, 0], [], [1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([1, 2, 1, 2, 1, 2], Vector{Bool}[[1, 0, 1, 0], [1, 0, 1, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([1, 2, 1, 2, 1, 2], Vector{Bool}[[1, 0, 1, 0], [1, 0, 1, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 4, T = [1, 0])])
 ([3, 3, 0, 0, 0, 3], Vector{Bool}[[1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], []], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 3, 0, 0, 0, 3], Vector{Bool}[[1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], []], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 10, T = [1, 0])])
 ([1, 3, 0, 0, 0, 3], Vector{Bool}[[1, 1, 0, 0], [1, 1, 0, 0], [1, 1, 0, 0, 1, 0], []], @NamedTuple{g

In [34]:
rest = remove_symmetries(setdiff(bezout, viro), symmetries)

3-element Vector{NWT}:
 ([0, 0, 0, 3, 1, 1], Vector{Bool}[[], [1, 0], [1, 1, 0, 0], [1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 3, T = [1, 0])])
 ([3, 0, 1, 0, 3, 0], Vector{Bool}[[1, 1, 0, 0], [1, 0, 1, 1, 0, 0], [], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 10, T = [1, 0])])
 ([0, 1, 3, 3, 0, 0], Vector{Bool}[[1, 1, 0, 0], [], [1, 1, 0, 0], [1, 0, 1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 8, T = [1, 0])])

In [35]:
viro2_dict = Dict{NWT,Tuple{Int,Vector{Bool}}}()
for (nwt, (itriang, sign)) in viro_dict
    nwt_e = forget_edges(
        nwt,
        lines_xz_3points,
        [1, 3, 4, 6],
        [
            [(false, 1, 1), (false, 3, 1), (false, 3, 3), (false, 1, 3)],
            [(false, 2, 1), (false, 4, 1), (false, 4, 3), (false, 2, 3)]])
    nwt_ev = forget_vertices(nwt_e, lines_xz, [[1, 3], [4, 2]])

    if !haskey(viro2_dict, nwt)
        viro2_dict[nwt_ev] = (itriang, sign)
    end
end

open(GzipCompressorStream, "cubic_2lines.jsonl.gz", "w") do out_io
    write_cc_to_jsonl!(out_io, lines_xz)
    for (nwt, (itriang, sign)) in viro2_dict
        write_nwt_to_jsonl!(out_io, nwt, itriang, sign)
    end
end

viro2 = collect(keys(viro2_dict))

46-element Vector{NWT}:
 ([1, 1], Vector{Bool}[[1, 0], [1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 1, T = [1, 0])])
 ([3, 1], Vector{Bool}[[1, 1, 0, 0], [1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 4, T = [1, 0])])
 ([3, 3], Vector{Bool}[[1, 1, 0, 1, 0, 0], [1, 0, 1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 3], Vector{Bool}[[1, 0, 1, 0, 1, 0], [1, 1, 0, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([1, 1], Vector{Bool}[[1, 0], [1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 4, T = [1, 0])])
 ([3, 3], Vector{Bool}[[1, 0, 1, 0, 1, 0], [1, 1, 1, 0, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 3], Vector{Bool}[[1, 1, 1, 0, 0, 0], [1, 0, 1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([1, 3], Vector{Bool}[[1, 1, 0, 0], [1, 1, 0, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[(g = 5, T = [1, 0])])
 ([1, 3], Vector{Bool}[[1, 1, 0, 0], [1, 0, 1, 0]], @NamedTuple{g::Int64, T::Vector{Bool}}[])
 ([3, 3], Vector{Bool}[[1, 0, 

In [36]:
bezout2 = NWT[]
for nwt in bezout
    nwt_e = forget_edges(
        nwt,
        lines_xz_3points,
        [1, 3, 4, 6],
        [
            [(false, 1, 1), (false, 3, 1), (false, 3, 3), (false, 1, 3)],
            [(false, 2, 1), (false, 4, 1), (false, 4, 3), (false, 2, 3)]])
    nwt_ev = forget_vertices(nwt_e, lines_xz, [[1, 3], [4, 2]])

    push!(bezout2, nwt_ev)
end
unique!(bezout2)
setdiff(bezout2, viro2)

NWT[]